# 高校数学とJulia言語 Day 5

- 城北中学校・高等学校　中学3年・高校1年
- 夏期講習会III 2026/8/24~2026/8/28
- 担当：清水団

## 本日のテーマ:確率とシミュレーション

### 今年のテーマ「実験する高校数学」

- 整数は**探す**・場合の数は**全部作る**・関数は**描く**・確率は**試す**

### 5日間の学習予定

- **Day 1**：Google Colabの紹介・Juliaで計算しよう ✅
- **Day 2**：整数問題をプログラムで考えよう ✅
- **Day 3**：場合の数・組合せを実験しよう ✅
- **Day 4**：関数・グラフ・最大最小 ✅
- **Day 5**：確率・シミュレーション ← 今日


## 今日のテーマ:「試す」

確率は「試してみる」ことができる数学です。

- コイン・サイコロを何万回も投げる（コンピュータなら一瞬!）
- 直感に反する「誕生日問題」「モンティ・ホール問題」
- 乱数で円周率を求める「モンテカルロ法」

最終日のゴール:**「理論」と「実験」が一致する瞬間**を自分の目で見ること。

## 必要なパッケージの準備

グラフの描画に必要なパッケージを読み込みましょう。

In [ ]:
# パッケージの読み込み
# フォント設定（日本語ラベルのため）
using Pkg
Pkg.add(url="https://github.com/ujimushi/PlotsGRBackendFontJaEmoji.jl")
using PlotsGRBackendFontJaEmoji, Plots
gr()

println("パッケージの読み込み完了!")

## コイン投げのシミュレーション

`rand()` は0以上1未満の乱数を返します。0.5未満なら「表」とすれば、コイン投げの完成です。

In [ ]:
# 乱数を出してみよう（実行するたびに変わる!）
rand()

In [ ]:
# コイン投げの関数を定義
coin_flip() = rand() < 0.5 ? "表" : "裏"

# 10回投げてみる
for i in 1:10
    println("$(i)回目: $(coin_flip())")
end

In [ ]:
# 1000回投げて表の回数を数える
n = 1000
omote = count(_ -> rand() < 0.5, 1:n)
println("表: $omote 回 / $n 回 → 割合 $(omote/n)")

In [ ]:
# 回数を増やすと 0.5 に近づく? 実験!
for n in [10, 100, 1000, 10000, 100000, 1000000]
    omote = count(_ -> rand() < 0.5, 1:n)
    println("$n 回 → 表の割合 $(omote/n)")
end

In [ ]:
# 割合の変化をグラフで見よう（大数の法則）
n = 10000
results = [rand() < 0.5 ? 1 : 0 for _ in 1:n]
ratios = cumsum(results) ./ (1:n)
plot(ratios, lw=1, label="表の割合", xlabel="投げた回数", ylabel="割合")
hline!([0.5], lw=2, color=:red, label="理論値 0.5")

最初は大きくブレますが、回数を増やすと理論値0.5に**収束**していきます。

これが**大数の法則**。教科書の言葉が、グラフで「見え」ました。

## サイコロのシミュレーション

`rand(1:6)` で1〜6の整数が等確率で出ます。

In [ ]:
# サイコロを10回投げる
rand(1:6, 10)

In [ ]:
# 60000回投げて、各目の回数を数える
n = 60000
rolls = rand(1:6, n)
for me in 1:6
    kaisu = count(==(me), rolls)
    println("$me の目: $kaisu 回（割合 $(round(kaisu/n, digits=4))）")
end
# 理論値は 1/6 ≈ 0.1667

In [ ]:
# 度数分布を棒グラフに
n = 60000
rolls = rand(1:6, n)
counts = [count(==(me), rolls) for me in 1:6]
bar(1:6, counts, label="実験値", xlabel="目", ylabel="回数")
hline!([n/6], lw=2, color=:red, label="理論値 n/6")

## 2つのサイコロの和

大小2つのサイコロの目の和は2〜12。どの和が出やすいでしょう?

**予想してから**実験しましょう!

In [ ]:
# 2つのサイコロの和を100000回実験
n = 100000
sums = [rand(1:6) + rand(1:6) for _ in 1:n]
counts = [count(==(s), sums) for s in 2:12]
bar(2:12, counts ./ n, label="実験値", xlabel="目の和", ylabel="確率")
scatter!(2:12, [min(s-1, 13-s)/36 for s in 2:12], color=:red, ms=6, label="理論値")

和が7になる確率が最大（$\frac{6}{36} = \frac{1}{6}$）。

昨日の「全部作る」で数えた $6$ 通り（1+6, 2+5, 3+4, 4+3, 5+2, 6+1）と、今日の実験がつながりました!

## 誕生日問題:直感 vs 確率

**問題**:40人のクラスで「同じ誕生日のペア」がいる確率は?

直感では低そうですが……実験してみましょう。

In [ ]:
# 40人の誕生日をランダムに決めて、かぶりがあるか調べる実験を10000回
function birthday_experiment(k)
    n_trials = 10000
    hits = count(1:n_trials) do _
        bd = rand(1:365, k)
        length(unique(bd)) < k   # かぶりがあればtrue
    end
    return hits / n_trials
end

birthday_experiment(40)

In [ ]:
# 理論値と比較:P(かぶりあり) = 1 - 365/365 × 364/365 × ...
function birthday_theory(k)
    p = 1.0
    for i in 0:k-1
        p *= (365 - i) / 365
    end
    return 1 - p
end

println("実験値: $(birthday_experiment(40))")
println("理論値: $(birthday_theory(40))")

In [ ]:
# 人数を変えて実験値と理論値を重ねてグラフに
ks = 1:60
plot(ks, [birthday_experiment(k) for k in ks], label="実験値", lw=2, xlabel="人数", ylabel="確率")
plot!(ks, [birthday_theory(k) for k in ks], label="理論値", lw=2, ls=:dash)
hline!([0.5], color=:gray, label="50%")

**23人**で確率50%超え、40人なら約**89%**!

直感は当てになりません。だからこそ、確率を「計算」し「実験」する価値があるのです。

## モンティ・ホール問題:世界一議論を呼んだ確率

3つの扉のうち1つに賞品。あなたが1つ選ぶと、司会者（正解を知っている）が残りからハズレの扉を開けます。

**「扉を変える」べき? 「変えない」べき?**

天才数学者たちも間違えたこの問題、実験なら一発です。

In [ ]:
# モンティ・ホール問題のシミュレーション
function monty_hall(change::Bool)
    doors = [1, 2, 3]
    atari = rand(doors)          # 当たりの扉
    choice = rand(doors)         # 最初の選択
    # 司会者は「当たりでも選択でもない」扉を開ける
    host = rand(setdiff(doors, [atari, choice]))
    if change
        choice = only(setdiff(doors, [choice, host]))  # 残った扉に変更
    end
    return choice == atari
end

n = 100000
println("変えない戦略の勝率: $(count(_ -> monty_hall(false), 1:n) / n)")
println("変える戦略の勝率:   $(count(_ -> monty_hall(true), 1:n) / n)")

実験結果:**変えると勝率が2倍**（$\frac{1}{3}$ → $\frac{2}{3}$）!

理屈で納得できなくても、10万回の実験結果は否定できません。「まず実験、それから理論」もアリなのです。

## モンテカルロ法:乱数で円周率を求める

正方形の中にランダムに点を打ち、円の中に入った点の割合から $\pi$ を求めます。

$$\frac{円の面積}{正方形の面積} = \frac{\pi}{4} \approx \frac{円内の点の数}{全部の点の数}$$

In [ ]:
# 点を2000個打って可視化
n = 2000
xs, ys = rand(n), rand(n)
inside = xs.^2 .+ ys.^2 .≤ 1
scatter(xs[inside], ys[inside], ms=2, color=:red, label="円の内側", aspect_ratio=:equal)
scatter!(xs[.!inside], ys[.!inside], ms=2, color=:blue, label="円の外側")

In [ ]:
# πの近似値を計算
function monte_carlo_pi(n)
    inside = count(_ -> rand()^2 + rand()^2 ≤ 1, 1:n)
    return 4 * inside / n
end

for n in [100, 10000, 1000000, 100000000]
    println("$n 個の点 → π ≈ $(monte_carlo_pi(n))")
end
println("本当のπ  = $π")

点を増やすほど $\pi = 3.14159...$ に近づきます。

**面積を「乱数」で求める**——この発想は、天気予報や金融、AIの世界で本当に使われています。

## 5日間のまとめ:「理論」と「実験」は一致する

| 日 | テーマ | アプローチ |
|----|--------|-----------|
| Day 1 | 計算・文法 | Juliaと友だちになる |
| Day 2 | 整数 | **探す** |
| Day 3 | 場合の数 | **全部作る** |
| Day 4 | 関数 | **描く** |
| Day 5 | 確率 | **試す** |

今日、コインの割合は0.5に収束し、誕生日問題の実験値は理論値とぴったり重なり、モンティ・ホールは$\frac{2}{3}$になりました。

**数学の理論は、実験で確かめられる。そして実験は、新しい理論の入り口になる。**

> Juliaは計算するための言語ではなく、**「数学を実験するための言語」**である。

これからも、気になる問題があったら「まず実験」してみてください。5日間お疲れさまでした!

## Day 5 の演習問題

- 予想 → 実験 → 理論の順で取り組んでみよう
- ノートブックを保存して、Google Classroomから提出しよう

### 問題1: コイン3枚

コインを3枚投げて「3枚とも表」になる確率を10000回の実験で求め、理論値 $\frac{1}{8}$ と比較しよう。

### 問題2: サイコロの最大値

サイコロを2個投げたときの「大きい方の目」の平均値を実験で求めよう。
（`max(rand(1:6), rand(1:6))` が使えます）

### 問題3: モンティ・ホール変形版

扉が**4つ**（当たり1つ、司会者がハズレを1つ開ける）の場合、変える戦略の勝率はどうなるか実験しよう。

### 問題4: 自由課題

自分で面白い確率の問題を考えて、シミュレーションで解いてみよう。

例:じゃんけんであいこになる確率 / くじ引きは何番目に引くと有利? / ランダムウォーク

## 解答欄

以下のセルに解答を記入してください。

### 問題1の解答

In [ ]:
# コイン3枚の実験



### 問題2の解答

In [ ]:
# 最大値の平均



### 問題3の解答

In [ ]:
# 扉4つのモンティ・ホール



### 問題4の解答

In [ ]:
# 自由課題

